In [48]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import nltk 
from nltk.corpus import stopwords
nltk.download('stopwords')
stop = set(stopwords.words('english'))

[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>


In [49]:
df = pd.read_csv(r"/Users/admin/Desktop/Open ME/Udemy Work/Toxic Comment Classification/data/train.csv")
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [50]:
# Check if there are any duplicate rows
print("The shape of the dataset before removing duplicates: ", df.shape)
df = df.drop_duplicates()
print("The shape of the dataset after removing duplicates: ", df.shape)

The shape of the dataset before removing duplicates:  (159571, 8)
The shape of the dataset after removing duplicates:  (159571, 8)


In [51]:
# Analysing the distribution of the labels
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
df['label_sum'] = df[label_cols].sum(axis=1)
df['label_sum'].value_counts()

label_sum
0    143346
1      6360
3      4209
2      3480
4      1760
5       385
6        31
Name: count, dtype: int64

In [52]:
# Print the number of comments in each category

for i in label_cols:
    print("The number of comments in the category ", i, " is: ", df[i].sum())

The number of comments in the category  toxic  is:  15294
The number of comments in the category  severe_toxic  is:  1595
The number of comments in the category  obscene  is:  8449
The number of comments in the category  threat  is:  478
The number of comments in the category  insult  is:  7877
The number of comments in the category  identity_hate  is:  1405


In [53]:
# Remove URL from the comment_text items
df['comment_text'] = df['comment_text'].str.replace('http\S+|www.\S+', '', case=False)

# Remove all numbers from the comment_text items
df['comment_text'] = df['comment_text'].str.replace('\d+', '', regex=True)

# Remove all stopwords from the comment_text items
df['comment_text'] = df['comment_text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))

# Remove special characters from the comment_text items
df['comment_text'] = df['comment_text'].str.replace('[^A-Za-z0-9]+', ' ', regex=True)

# Remove any HTML tags from the comment_text items without using bs4
df['comment_text'] = df['comment_text'].str.replace('<.*?>', '', regex=True)

# Convert all items in the comment_text column to lowercase
df['comment_text'] = df['comment_text'].str.lower()

In [54]:
# Tokenising the comment_text items

from nltk.tokenize import word_tokenize
df['comment_text'] = df['comment_text'].apply(word_tokenize)

df['comment_text'][2]

['hey',
 'man',
 'i',
 'm',
 'really',
 'trying',
 'edit',
 'war',
 'it',
 's',
 'guy',
 'constantly',
 'removing',
 'relevant',
 'information',
 'talking',
 'edits',
 'instead',
 'talk',
 'page',
 'he',
 'seems',
 'care',
 'formatting',
 'actual',
 'info']

In [55]:
# In each item in comment_text, remove any empty strings, strings containing only one character, and strings containing only digits
df['comment_text'] = df['comment_text'].apply(lambda x: [item for item in x if len(item) > 1 and not item.isdigit()])
df['comment_text'][2]

['hey',
 'man',
 'really',
 'trying',
 'edit',
 'war',
 'it',
 'guy',
 'constantly',
 'removing',
 'relevant',
 'information',
 'talking',
 'edits',
 'instead',
 'talk',
 'page',
 'he',
 'seems',
 'care',
 'formatting',
 'actual',
 'info']

In [56]:
# Perform Lemmatization on the comment_text items
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
df['comment_text'] = df['comment_text'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

df['comment_text'][2]

['hey',
 'man',
 'really',
 'trying',
 'edit',
 'war',
 'it',
 'guy',
 'constantly',
 'removing',
 'relevant',
 'information',
 'talking',
 'edits',
 'instead',
 'talk',
 'page',
 'he',
 'seems',
 'care',
 'formatting',
 'actual',
 'info']

In [57]:
# Remove all stopwords from the comment_text items
df['comment_text'] = df['comment_text'].apply(lambda x: ' '.join([word for word in x if word not in (stop)]))

df['comment_text'][2]

'hey man really trying edit war guy constantly removing relevant information talking edits instead talk page seems care formatting actual info'

In [58]:
# Perform tokenization on the comment_text items
df['comment_text'] = df['comment_text'].apply(word_tokenize)

df['comment_text'][2]

['hey',
 'man',
 'really',
 'trying',
 'edit',
 'war',
 'guy',
 'constantly',
 'removing',
 'relevant',
 'information',
 'talking',
 'edits',
 'instead',
 'talk',
 'page',
 'seems',
 'care',
 'formatting',
 'actual',
 'info']

In [59]:
# Find the item with the largest length in comment_text
max_length = max(df['comment_text'].apply(len))

print("The item with the largest length in comment_text is: ", df[df['comment_text'].apply(len) == max_length]['comment_text'].values[0])
print("The length of the item with the largest length in comment_text is: ", max_length)

The item with the largest length in comment_text is:  ['pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 'pig', 

In [60]:
import gensim
from sklearn.manifold import TSNE

import gensim.downloader as api

In [61]:
import gensim

# Path to the extracted GloVe file
glove_path = "/Users/admin/gensim-data/glove-twitter/glove.twitter.27B.25d.txt"

# Load GloVe as a Word2Vec model
glove_model = gensim.models.KeyedVectors.load_word2vec_format(glove_path, binary=False, no_header=True)

# Test the model
print(glove_model.most_similar("hello"))

[('thanks', 0.9292947053909302), ('thank', 0.9241276383399963), ('birthday', 0.9218813180923462), ('welcome', 0.919857382774353), ('happy', 0.9158351421356201), ('hey', 0.9121533632278442), ('miss', 0.9077523946762085), ('love', 0.9076114296913147), ('dear', 0.8950147032737732), ('babe', 0.8825745582580566)]


In [62]:
glove_model.most_similar("obscene")

[('egregious', 0.8912188410758972),
 ('arbitrary', 0.8795605897903442),
 ('over-the-top', 0.8781465888023376),
 ('impactful', 0.8736109137535095),
 ('symbolic', 0.8712402582168579),
 ('dubious', 0.8708045482635498),
 ('unsavory', 0.8605940937995911),
 ('menacing', 0.8546699285507202),
 ('innocuous', 0.854274332523346),
 ('objectionable', 0.8521093726158142)]

In [63]:
# Use GloVe to convert the comment_text items to vectors
# Note that each item in comment_text is a list of words
# We will have a list of lists of vectors
vectors = []
for comment in df['comment_text']:
    comment_vectors = []
    for word in comment:
        try:
            vector = glove_model[word]
            comment_vectors.append(vector)
        except:
            pass
    vectors.append(comment_vectors)

# Make sure that the number of vectors in each item of comment_text is equal to the number of words in that item
print("The number of items in vectors is: ", len(vectors))
print("The number of items in comment_text is: ", len(df['comment_text']))

# Append the vectors to the dataframe
df['vectors'] = vectors



The number of items in vectors is:  159571
The number of items in comment_text is:  159571


In [64]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,label_sum,vectors
0,0000997932d777bf,"[explanation, edits, made, username, hardcore,...",0,0,0,0,0,0,0,"[[0.64556, 0.017024, -0.9843, 0.82822, 0.4259,..."
1,000103f0d9cfb60f,"[aww, match, background, colour, seemingly, st...",0,0,0,0,0,0,0,"[[-0.67699, -0.37985, 0.54968, 0.11194, -0.521..."
2,000113f07ec002fd,"[hey, man, really, trying, edit, war, guy, con...",0,0,0,0,0,0,0,"[[-0.011587, 0.6575, 0.19401, 0.1987, -0.69354..."
3,0001b41b1c6bb37e,"[make, real, suggestion, improvement, wondered...",0,0,0,0,0,0,0,"[[0.27309, 0.60972, -0.50972, 0.15691, -0.3243..."
4,0001d958c54c6e35,"[sir, hero, chance, remember, page]",0,0,0,0,0,0,0,"[[-0.31768, -0.031633, 0.041958, -0.19124, -0...."


In [65]:
import numpy as np

# Function to average the word vectors in each comment
def average_vectors(vectors):
    return np.mean(vectors, axis=0)

# Apply the function to the vectors column to create a new column of averaged embeddings
df['embedding'] = df['vectors'].apply(average_vectors)



/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [66]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,label_sum,vectors,embedding
0,0000997932d777bf,"[explanation, edits, made, username, hardcore,...",0,0,0,0,0,0,0,"[[0.64556, 0.017024, -0.9843, 0.82822, 0.4259,...","[-0.009584394, 0.2666462, 0.12415261, -0.19998..."
1,000103f0d9cfb60f,"[aww, match, background, colour, seemingly, st...",0,0,0,0,0,0,0,"[[-0.67699, -0.37985, 0.54968, 0.11194, -0.521...","[-0.21464959, -0.006039864, -0.010166699, -0.3..."
2,000113f07ec002fd,"[hey, man, really, trying, edit, war, guy, con...",0,0,0,0,0,0,0,"[[-0.011587, 0.6575, 0.19401, 0.1987, -0.69354...","[0.16683006, 0.36873612, -0.03913395, -0.20699..."
3,0001b41b1c6bb37e,"[make, real, suggestion, improvement, wondered...",0,0,0,0,0,0,0,"[[0.27309, 0.60972, -0.50972, 0.15691, -0.3243...","[0.19912972, 0.24817635, -0.08939304, -0.28691..."
4,0001d958c54c6e35,"[sir, hero, chance, remember, page]",0,0,0,0,0,0,0,"[[-0.31768, -0.031633, 0.041958, -0.19124, -0....","[0.11673601, 0.13099541, 0.0798596, -0.0819, -..."


In [67]:
df['embedding'][4]

array([ 0.11673601,  0.13099541,  0.0798596 , -0.0819    , -0.2058852 ,
       -0.502638  ,  0.9692861 ,  0.2815788 , -0.1515    , -0.622016  ,
       -0.189168  ,  0.18814799, -3.62532   ,  0.137472  ,  0.0533982 ,
        0.159866  ,  0.1515969 , -0.276162  , -0.53356236, -0.20170519,
       -0.16246761, -0.07755999, -0.357703  ,  0.04895501, -0.70491004],
      dtype=float32)

In [68]:
# In each list in embedding column check if there are any NaN values
is_na_values = df['embedding'].apply(lambda x: np.isnan(x).sum())
print("The number of NaN values in the embedding column is: ", is_na_values.sum())

# Print index of any embedding item that contains NaN values and the item itself
print("The index of the embedding item that contains NaN values is: ", is_na_values[is_na_values > 0].index[0])

The number of NaN values in the embedding column is:  139
The index of the embedding item that contains NaN values is:  1050


In [69]:
# For all rows containing NaN, replace it with zero vector of length 25
df['embedding'] = df['embedding'].apply(lambda x: np.zeros(25) if np.isnan(x).sum() > 0 else x)


In [70]:
# In each list in embedding column check if there are any NaN values
is_na_values = df['embedding'].apply(lambda x: np.isnan(x).sum())
print("The number of NaN values in the embedding column is: ", is_na_values.sum())

The number of NaN values in the embedding column is:  0


In [71]:
# Check if each embedding item has the same length
embedding_lengths = df['embedding'].apply(len)
print("The number of unique lengths in the embedding column is: ", embedding_lengths.nunique())

The number of unique lengths in the embedding column is:  1


In [72]:
from keras.models import Sequential
from keras.layers import Dense, Dropout

# Define the model
model = Sequential()
model.add(Dense(256, input_dim=25, activation='relu'))  # Assuming GloVe embeddings are 100-dimensional
model.add(Dropout(0.5))  # Dropout layer to prevent overfitting
model.add(Dense(128, activation='relu'))
model.add(Dense(6, activation='sigmoid'))  # 5 outputs, one for each label

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Model summary
model.summary()


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 256)            │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,326 (157.52 KB)

 Trainable params: 40,326 (157.52 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
print(np.array(df['embedding'].tolist()).shape)


(159571, 25)


In [74]:
X = np.array(df['embedding'].tolist())
print(X.shape)  # Ensure this is (num_samples, 100)


(159571, 25)


In [75]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,label_sum,vectors,embedding
0,0000997932d777bf,"[explanation, edits, made, username, hardcore,...",0,0,0,0,0,0,0,"[[0.64556, 0.017024, -0.9843, 0.82822, 0.4259,...","[-0.009584394, 0.2666462, 0.12415261, -0.19998..."
1,000103f0d9cfb60f,"[aww, match, background, colour, seemingly, st...",0,0,0,0,0,0,0,"[[-0.67699, -0.37985, 0.54968, 0.11194, -0.521...","[-0.21464959, -0.006039864, -0.010166699, -0.3..."
2,000113f07ec002fd,"[hey, man, really, trying, edit, war, guy, con...",0,0,0,0,0,0,0,"[[-0.011587, 0.6575, 0.19401, 0.1987, -0.69354...","[0.16683006, 0.36873612, -0.03913395, -0.20699..."
3,0001b41b1c6bb37e,"[make, real, suggestion, improvement, wondered...",0,0,0,0,0,0,0,"[[0.27309, 0.60972, -0.50972, 0.15691, -0.3243...","[0.19912972, 0.24817635, -0.08939304, -0.28691..."
4,0001d958c54c6e35,"[sir, hero, chance, remember, page]",0,0,0,0,0,0,0,"[[-0.31768, -0.031633, 0.041958, -0.19124, -0....","[0.11673601, 0.13099541, 0.0798596, -0.0819, -..."


In [76]:
X = np.array(df['embedding'].tolist())
y = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values

model.fit(X, y, epochs=100, batch_size=32, validation_split=0.2)



Epoch 1/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9532 - loss: 0.0939 - val_accuracy: 0.9941 - val_loss: 0.0779
Epoch 2/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9866 - loss: 0.0760 - val_accuracy: 0.9911 - val_loss: 0.0754
Epoch 3/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9809 - loss: 0.0763 - val_accuracy: 0.9941 - val_loss: 0.0752
Epoch 4/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9903 - loss: 0.0747 - val_accuracy: 0.9937 - val_loss: 0.0761
Epoch 5/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9903 - loss: 0.0749 - val_accuracy: 0.9941 - val_loss: 0.0738
Epoch 6/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9909 - loss: 0.0743 - val_accuracy: 0.9941 - val_loss: 0.0741
Epoch 7/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9886 - loss: 0.0734 - val_accuracy: 0.9933 - val_loss: 0.0735
Epoch 8/100
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9882 - loss: 0

In [77]:
# Evaluate the model
loss, accuracy = model.evaluate(np.array(df['embedding'].tolist()), y)
print(f'Loss: {loss}, Accuracy: {accuracy}')

4987/4987 ━━━━━━━━━━━━━━━━━━━━ 4s 823us/step - accuracy: 0.9912 - loss: 0.0634
Loss: 0.06523998081684113, Accuracy: 0.990844190120697


In [91]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

# Define the LSTM model
model = Sequential()
model.add(LSTM(128, input_shape=(25, 100), return_sequences=True))  # First LSTM layer
model.add(Dropout(0.5))  # Dropout for regularization
model.add(LSTM(64, return_sequences=False))  # Second LSTM layer
model.add(Dropout(0.5))  # Dropout for regularization
model.add(Dense(6, activation='sigmoid'))  # 6 output labels (multi-label classification)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 25, 128)        │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 25, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 167,046 (652.52 KB)

 Trainable params: 167,046 (652.52 KB)

 Non-trainable params: 0 (0.00 B)

In [97]:
X = np.array(df['embedding'].tolist())
y = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values

model.fit(X, y, epochs=20, batch_size=32, validation_split=0.2)

Epoch 1/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 1365s 342ms/step - accuracy: 0.8081 - loss: 0.1423 - val_accuracy: 0.9941 - val_loss: 0.0935
Epoch 2/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 1228s 308ms/step - accuracy: 0.9921 - loss: 0.0922 - val_accuracy: 0.9941 - val_loss: 0.0821
Epoch 3/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 125s 31ms/step - accuracy: 0.9941 - loss: 0.0826 - val_accuracy: 0.9941 - val_loss: 0.0781
Epoch 4/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 124s 31ms/step - accuracy: 0.9942 - loss: 0.0808 - val_accuracy: 0.9941 - val_loss: 0.0815
Epoch 5/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 125s 31ms/step - accuracy: 0.9940 - loss: 0.0800 - val_accuracy: 0.9941 - val_loss: 0.0789
Epoch 6/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 124s 31ms/step - accuracy: 0.9934 - loss: 0.0789 - val_accuracy: 0.9941 - val_loss: 0.0781
Epoch 7/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 129s 32ms/step - accuracy: 0.9940 - loss: 0.0798 - val_accuracy: 0.9941 - val_loss: 0.0766
Epoch 8/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 131s 33ms/step - accura

In [98]:
# Evaluate the model
loss, accuracy = model.evaluate(np.array(df['embedding'].tolist()), y)
print(f'Loss: {loss}, Accuracy: {accuracy}')

4987/4987 ━━━━━━━━━━━━━━━━━━━━ 58s 12ms/step - accuracy: 0.9943 - loss: 0.0725
Loss: 0.07356993854045868, Accuracy: 0.9941655993461609


# Testing with test.csv

In [78]:
# Take the test data and preprocess it in the same way as the training data
df_test = pd.read_csv(r"/Users/admin/Desktop/Open ME/Udemy Work/Toxic Comment Classification/data/test.csv")

# Remove URL from the comment_text items
df_test['comment_text'] = df_test['comment_text'].str.replace('http\S+|www.\S+', '', case=False)

# Remove all numbers from the comment_text items
df_test['comment_text'] = df_test['comment_text'].str.replace('\d+', '', regex=True)

# Remove all stopwords from the comment_text items
df_test['comment_text'] = df_test['comment_text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))

# Remove special characters from the comment_text items
df_test['comment_text'] = df_test['comment_text'].str.replace('[^A-Za-z0-9]+', ' ', regex=True)

# Remove any HTML tags from the comment_text items without using bs4
df_test['comment_text'] = df_test['comment_text'].str.replace('<.*?>', '', regex=True)

# Convert all items in the comment_text column to lowercase
df_test['comment_text'] = df_test['comment_text'].str.lower()


In [79]:
# Tokenising the comment_text items

from nltk.tokenize import word_tokenize
df_test['comment_text'] = df_test['comment_text'].apply(word_tokenize)

# In each item in comment_text, remove any empty strings, strings containing only one character, and strings containing only digits
df_test['comment_text'] = df_test['comment_text'].apply(lambda x: [item for item in x if len(item) > 1 and not item.isdigit()])

# Perform Lemmatization on the comment_text items
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
df_test['comment_text'] = df_test['comment_text'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])



In [80]:
# Remove all stopwords from the comment_text items
df_test['comment_text'] = df_test['comment_text'].apply(lambda x: ' '.join([word for word in x if word not in (stop)]))

# Perform tokenization on the comment_text items
df_test['comment_text'] = df_test['comment_text'].apply(word_tokenize)

In [81]:
# Use GloVe to convert the comment_text items to vectors
# Note that each item in comment_text is a list of words
# We will have a list of lists of vectors

vectors = []
for comment in df_test['comment_text']:
    comment_vectors = []
    for word in comment:
        try:
            vector = glove_model[word]
            comment_vectors.append(vector)
        except:
            pass
    vectors.append(comment_vectors)

# Make sure that the number of vectors in each item of comment_text is equal to the number of words in that item
print("The number of items in vectors is: ", len(vectors))
print("The number of items in comment_text is: ", len(df_test['comment_text']))

# Append the vectors to the dataframe
df_test['vectors'] = vectors



The number of items in vectors is:  153164
The number of items in comment_text is:  153164


In [82]:
# Function to average the word vectors in each comment
def average_vectors(vectors):
    return np.mean(vectors, axis=0)

# Apply the function to the vectors column to create a new column of averaged embeddings
df_test['embedding'] = df_test['vectors'].apply(average_vectors)

df_test.head()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,id,comment_text,vectors,embedding
0,00001cee341fdb12,"[yo, bitch, ja, rule, succesful, ever, whats, ...","[[0.37657, 1.2598, 0.41847, 0.8194, -0.36799, ...","[0.13327365, 0.4038652, 0.20251326, 0.00779432..."
1,0000247867823ef7,"[rfc, title, fine, imo]","[[-0.052976, 0.4181, -0.43597, -1.2931, -0.862...","[0.134176, 0.2065425, -0.22604938, -0.8456725,..."
2,00013b17ad220c46,"[source, zawe, ashton, lapland]","[[0.45217, -0.08378, -1.3449, 0.10053, 0.54661...","[-0.65314335, -0.27523667, -0.25312567, -0.347..."
3,00017563c3f7919a,"[look, back, source, information, updated, cor...","[[-0.21933, -0.5862, 0.32648, -0.27618, 0.3515...","[0.23391865, 0.31624934, -0.39171457, -0.26818..."
4,00017695ad8997eb,"[anonymously, edit, article]","[[0.090477, 0.46889, -0.42059, 0.28922, 0.0161...","[0.439509, 0.13595134, -0.31362998, 0.02680957..."


In [83]:
# Check if any of the embeddings contain NaN values
is_na_values = df_test['embedding'].apply(lambda x: np.isnan(x).sum())
print("The number of NaN values in the embedding column is: ", is_na_values.sum())

The number of NaN values in the embedding column is:  1520


In [84]:
# For the rows that contain NaN values in the embedding column, convert them into a list of zeros
df_test['embedding'] = df_test['embedding'].apply(lambda x: np.zeros(25) if np.isnan(x).sum() > 0 else x)


In [85]:
# Find the length of the each item in the embedding column and check if they are all the same
embedding_lengths = df_test['embedding'].apply(len)
print("The number of unique lengths in the embedding column is: ", embedding_lengths.nunique())

The number of unique lengths in the embedding column is:  1


In [99]:
# Run the model on the test data
X_test = np.array(df_test['embedding'].tolist())
y_pred = model.predict(X_test)

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step


In [87]:
# See the sample submission file
sample_submission = pd.read_csv(r"/Users/admin/Desktop/Open ME/Udemy Work/Toxic Comment Classification/data/sample_submission.csv")
sample_submission.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.5,0.5,0.5,0.5,0.5,0.5
1,0000247867823ef7,0.5,0.5,0.5,0.5,0.5,0.5
2,00013b17ad220c46,0.5,0.5,0.5,0.5,0.5,0.5
3,00017563c3f7919a,0.5,0.5,0.5,0.5,0.5,0.5
4,00017695ad8997eb,0.5,0.5,0.5,0.5,0.5,0.5


In [100]:
# Convert the predictions into a dataframe with the correct column names and like the sample submission file
df_submission = pd.DataFrame(df_test['id'])
df_submission = pd.concat([df_submission, pd.DataFrame(y_pred, columns=['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'])], axis=1)
df_submission.head()



,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.991752,3.539695e-01,0.955386,4.090821e-02,0.853220,2.149454e-01
1,0000247867823ef7,0.019736,2.929511e-05,0.004020,8.854965e-05,0.004035,5.973168e-04
2,00013b17ad220c46,0.007661,3.937665e-06,0.001284,1.233035e-05,0.001466,4.823890e-04
3,00017563c3f7919a,0.000244,7.690039e-09,0.000039,8.537417e-08,0.000029,5.103719e-07
4,00017695ad8997eb,0.008297,6.106643e-06,0.001627,1.943456e-05,0.001725,1.479489e-04


In [101]:
# Save the dataframe as a CSV file with name 'submission_advanced_model.csv'
df_submission.to_csv(r"/Users/admin/Desktop/Open ME/Udemy Work/Toxic Comment Classification/data/submission_advanced_model.csv", index=False)